# 🔥 Edge-AI Wildfire Spatiotemporal Forecasting System
## Exploratory Data Analysis, Model Training & Evaluation Notebook

**Project:** Proactive 24–72h Wildfire Risk Forecasting using Stacked ConvLSTM2D and ST-GNN  
**Dataset:** Synthetic Spatiotemporal Data Cubes (physics-based simulation) + Google Earth Engine (Production)  
**Alignment:** UN SDG 13 (Climate Action) · UN SDG 15 (Life on Land)

---

### 🌍 Data Sources Used

| Channel | Variable | Production Source (Google Earth Engine) | GEE Collection ID |
|---|---|---|---|
| Ch 0 | Active Fire Mask | **NASA FIRMS** – VIIRS/MODIS Thermal Hotspots | `FIRMS` |
| Ch 1 | Wind Vector U (E-W) | **ECMWF ERA5-Land** Daily Aggregates | `ECMWF/ERA5_LAND/DAILY_AGGR` |
| Ch 2 | Wind Vector V (N-S) | **ECMWF ERA5-Land** Daily Aggregates | `ECMWF/ERA5_LAND/DAILY_AGGR` |
| Ch 3 | Relative Humidity | **ECMWF ERA5-Land** Surface humidity | `ECMWF/ERA5_LAND/DAILY_AGGR` |
| Ch 4 | Surface Temperature | **ECMWF ERA5-Land** Temperature 2m | `ECMWF/ERA5_LAND/DAILY_AGGR` |
| Ch 5 | NDVI (Fuel Density) | **MODIS MOD13A2** 16-Day 250m NDVI | `MODIS/061/MOD13A2` |
| Ch 6 | Terrain Slope | **USGS SRTM** Digital Elevation Model | `USGS/SRTMGL1_003` |

> **For this notebook:** We use a physically realistic synthetic data cube generator that simulates all 7 channels based on coupled fire-weather-vegetation dynamics — enabling full offline training and experimentation without GEE authentication.


## Section 1: Setup & Imports

In [ ]:
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import torch

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

from src.data.dataset_generator import WildfireDataCubeGenerator
from src.data.wildfire_dataset import get_wildfire_dataloaders
from src.models.convlstm import ConvLSTM2D
from src.models.st_gnn import SpatiotemporalGNN
from src.models.losses import CombinedFocalDiceLoss
from src.evaluation.metrics import compute_horizon_metrics, compute_iou, compute_pr_auc

plt.style.use('dark_background')
FIRE_CMAP = plt.cm.YlOrRd
print('All modules imported successfully.')

---
## Section 2: Dataset — Spatiotemporal Data Cube Exploration
### 2.1 — Generate the Dataset

In [ ]:
generator = WildfireDataCubeGenerator(height=128, width=128, seed=42)
X, Y = generator.create_dataset_samples(sequence_length=5, num_days=90)

print(f"Input Tensor X (Data Cube):  {X.shape}")
print(f"  Meaning: [N samples, T=5 days, C=7 channels, H=128, W=128 pixels]")
print(f"Target Tensor Y (Labels):    {Y.shape}")
print(f"  Meaning: [N samples, 3 horizons (24h/48h/72h), H=128, W=128 pixels]")
print()
print(f"Total Dataset Samples: {len(X)}")
fire_ratio = (Y > 0.5).float().mean().item()
print(f"Fire Pixel Ratio (class imbalance): {fire_ratio*100:.2f}% fire vs {(1-fire_ratio)*100:.2f}% background")
print(f"  → Severe imbalance: justifies Focal + Dice Loss combination")

### 2.2 — Visualize a 5-Day Input Data Cube (All 7 Channels)

In [ ]:
channel_names = [
    'Active Fire Mask\n(NASA FIRMS)',
    'Wind Vector U\n(ERA5-Land E/W)',
    'Wind Vector V\n(ERA5-Land N/S)',
    'Relative Humidity\n(ERA5-Land)',
    'Surface Temp\n(ERA5-Land)',
    'NDVI Fuel Density\n(MODIS MOD13A2)',
    'Terrain Slope\n(USGS SRTM DEM)'
]
cmaps = ['Reds', 'RdBu_r', 'RdBu_r', 'Blues_r', 'plasma', 'YlGn', 'Greys']

sample_idx = 5
sample = X[sample_idx]  # [T=5, C=7, H, W]

fig, axes = plt.subplots(5, 7, figsize=(21, 14))
fig.suptitle(f'5-Day Spatiotemporal Input Data Cube — Sample #{sample_idx}\n'
             f'Rows = Time Steps (Day 1→5) | Columns = Channels (7 environmental variables)',
             fontsize=13, fontweight='bold', y=1.01)

for t in range(5):
    for c in range(7):
        ax = axes[t, c]
        im = ax.imshow(sample[t, c].numpy(), cmap=cmaps[c], aspect='auto')
        ax.set_xticks([]); ax.set_yticks([])
        if t == 0:
            ax.set_title(channel_names[c], fontsize=8, pad=4)
        if c == 0:
            ax.set_ylabel(f'Day {t+1}', fontsize=9, rotation=90, labelpad=4)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('../notebooks/figures/data_cube_visualization.png', dpi=100, bbox_inches='tight')
plt.show()

### 2.3 — Visualize Ground Truth Labels: 24h, 48h, 72h Fire Spread Targets

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(18, 10))
fig.suptitle('Wildfire Spread Ground Truth Labels\n'
             'Rows = Forecast Horizon | Columns = 5 different spatial scenarios', fontsize=13, fontweight='bold')

horizon_labels = ['24h Forecast Target (+1 Day)', '48h Forecast Target (+2 Days)', '72h Forecast Target (+3 Days)']
sample_indices = [0, 2, 5, 8, 12]

for h_idx in range(3):
    for s_pos, s_idx in enumerate(sample_indices):
        ax = axes[h_idx, s_pos]
        fire_grid = Y[s_idx, h_idx].numpy()
        im = ax.imshow(fire_grid, cmap='Reds', vmin=0, vmax=1)
        ax.set_title(f'Sample #{s_idx}', fontsize=8)
        ax.set_xticks([]); ax.set_yticks([])
        fire_pct = (fire_grid > 0.5).sum() / fire_grid.size * 100
        ax.set_xlabel(f'{fire_pct:.1f}% fire pixels', fontsize=7, color='#FC8181')
        if s_pos == 0:
            ax.set_ylabel(horizon_labels[h_idx], fontsize=8, rotation=90, labelpad=5)

plt.tight_layout()
plt.savefig('../notebooks/figures/ground_truth_labels.png', dpi=100, bbox_inches='tight')
plt.show()

### 2.4 — Class Imbalance Analysis

In [ ]:
Y_np = Y.numpy()
fire_pcts_per_horizon = []
for h in range(3):
    pct = (Y_np[:, h, :, :] > 0.5).mean() * 100
    fire_pcts_per_horizon.append(pct)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart of class imbalance per horizon
ax1 = axes[0]
bars = ax1.bar(['24h Horizon', '48h Horizon', '72h Horizon'],
               fire_pcts_per_horizon, color=['#FC8181', '#F6AD55', '#FBD38D'], edgecolor='white')
ax1.set_ylabel('Fire Pixel Percentage (%)', fontsize=10)
ax1.set_title('Class Imbalance per Forecast Horizon\n(Fire vs Background Pixel Ratio)', fontsize=11, fontweight='bold')
for bar, pct in zip(bars, fire_pcts_per_horizon):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{pct:.2f}%', ha='center', va='bottom', fontsize=10, color='white')
ax1.set_ylim(0, max(fire_pcts_per_horizon) * 1.5)

# Pixel distribution histogram for 48h horizon
ax2 = axes[1]
values = Y_np[:, 1, :, :].ravel()
ax2.hist(values[values < 0.5], bins=30, color='#4FD1C5', alpha=0.7, label='Background pixels (0)')
ax2.hist(values[values >= 0.5], bins=30, color='#FC8181', alpha=0.9, label='Fire pixels (1)')
ax2.set_xlabel('Pixel Value', fontsize=10)
ax2.set_ylabel('Count', fontsize=10)
ax2.set_title('Pixel Value Distribution (48h Horizon)\n→ Justifies Focal+Dice Loss', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_yscale('log')

plt.tight_layout()
plt.savefig('../notebooks/figures/class_imbalance.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'\n📊 Class Imbalance Summary:')
for h, pct in zip(['24h', '48h', '72h'], fire_pcts_per_horizon):
    print(f'  Horizon {h}: {pct:.2f}% fire pixels | {100-pct:.2f}% background')
print(f'  → This severe imbalance (~{np.mean(fire_pcts_per_horizon):.1f}% positive class) ')
print(f'    requires Focal Loss (gamma=2, alpha=0.85) + Dice Loss to avoid trivial all-zero predictions.')

### 2.5 — Train / Val / Test Dataset Split Breakdown

In [ ]:
loaders = get_wildfire_dataloaders(batch_size=4, seq_len=5, total_days=90, seed=42)
shapes = loaders['shapes']

print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('         DATASET SPLIT SUMMARY')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'  Input Shape X:   {shapes["X_shape"]}')
print(f'  Target Shape Y:  {shapes["Y_shape"]}')
print()
total = shapes['train_len'] + shapes['val_len'] + shapes['test_len']
print(f'  Training Set:    {shapes["train_len"]} samples  ({shapes["train_len"]/total*100:.0f}%)  ← Model learns weights')
print(f'  Validation Set:  {shapes["val_len"]} samples   ({shapes["val_len"]/total*100:.0f}%)   ← Hyperparameter tuning')
print(f'  Test Set:        {shapes["test_len"]} samples   ({shapes["test_len"]/total*100:.0f}%)   ← Final evaluation only')
print(f'  Total Samples:   {total}')
print()
print('  ⚠️  CHRONOLOGICAL SPLIT — no temporal data leakage')
print('  Train [Day 1→63] → Val [Day 63→76] → Test [Day 76→90]')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

# Visual split bar
fig, ax = plt.subplots(figsize=(12, 1.2))
ax.barh(0, shapes['train_len'], color='#68D391', label=f'Train ({shapes["train_len"]})')
ax.barh(0, shapes['val_len'], left=shapes['train_len'], color='#F6AD55', label=f'Val ({shapes["val_len"]})')
ax.barh(0, shapes['test_len'], left=shapes['train_len']+shapes['val_len'], color='#FC8181', label=f'Test ({shapes["test_len"]})')
ax.set_xlim(0, total)
ax.set_yticks([])
ax.set_xlabel('Sample Index (chronological time →)', fontsize=10)
ax.set_title('Dataset Split: Chronological Train / Val / Test', fontsize=11, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('../notebooks/figures/dataset_split.png', dpi=100, bbox_inches='tight')
plt.show()

---
## Section 3: Model Architecture Overview

In [ ]:
convlstm = ConvLSTM2D(in_channels=7, hidden_dims=[32, 16], out_horizons=3)
st_gnn   = SpatiotemporalGNN(in_channels=7, patch_size=4, hidden_dim=32, out_horizons=3)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

dummy = torch.randn(1, 5, 7, 128, 128)
with torch.no_grad():
    out_cl  = convlstm(dummy)
    out_gnn = st_gnn(dummy)

print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('              MODEL ARCHITECTURE SUMMARY')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print()
print('🟢 Model 1: Stacked ConvLSTM2D (Primary)')
print(f'   Architecture: 2-Layer ConvLSTM (hidden=[32,16]) + Conv2D Head')
print(f'   Input:        [Batch, T=5, C=7, H=128, W=128]')
print(f'   Output:       {list(out_cl.shape)}  (3 risk grids: 24h, 48h, 72h)')
print(f'   Parameters:   {count_params(convlstm):,}')
print()
print('🔵 Model 2: Spatiotemporal Graph NN (ST-GNN) (Comparison)')
print(f'   Architecture: Patch Encoder → Wind-Weighted GraphConv → GRU → Decoder')
print(f'   Input:        [Batch, T=5, C=7, H=128, W=128]')
print(f'   Output:       {list(out_gnn.shape)}  (3 risk grids: 24h, 48h, 72h)')
print(f'   Parameters:   {count_params(st_gnn):,}')
print()
print('📌 Loss Function: CombinedFocalDiceLoss')
print('   L = 0.6 × FocalLoss(α=0.85, γ=2.0) + 0.4 × DiceLoss(smooth=1.0)')
print('   Focal Loss: Downweights easy background pixels, focuses on rare fire pixels')
print('   Dice Loss: Optimizes spatial overlap between predicted and actual fire masks')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

---
## Section 4: Model Training
### 4.1 — Run Training (or Load Saved History)

In [ ]:
import time

history_path = '../checkpoints/training_summary.json'

if os.path.exists(history_path):
    with open(history_path) as f:
        saved = json.load(f)
    hist_cl  = saved.get('convlstm', {}).get('history', {})
    hist_gnn = saved.get('st_gnn',   {}).get('history', {})
    print('✅ Loaded saved training history from checkpoints/training_summary.json')
    print(f'   ConvLSTM2D epochs trained: {len(hist_cl.get("train_loss", []))}')
    print(f'   ST-GNN epochs trained:     {len(hist_gnn.get("train_loss", []))}')
else:
    print('⚠️  No saved history found. Running quick 5-epoch demo training...')
    print('   (Run train.py from the project root for full 10-epoch training)')

    from src.models.losses import CombinedFocalDiceLoss
    from src.evaluation.metrics import compute_iou, compute_pr_auc, compute_pixel_accuracy

    def run_training(model, name, train_loader, val_loader, epochs=5, lr=1e-3):
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
        criterion = CombinedFocalDiceLoss()
        history   = {'train_loss': [], 'val_loss': [], 'val_iou': [],
                     'val_accuracy': [], 'val_f1': [], 'val_precision': [], 'val_recall': [], 'val_pr_auc': [], 'val_balanced_accuracy': []}
        for epoch in range(1, epochs + 1):
            model.train()
            tl = 0
            for bx, by in train_loader:
                optimizer.zero_grad()
                loss = criterion(model(bx), by)
                loss.backward()
                optimizer.step()
                tl += loss.item() * bx.size(0)
            tl /= len(train_loader.dataset)

            model.eval()
            vl, preds, tgts = 0, [], []
            with torch.no_grad():
                for bx, by in val_loader:
                    loss = criterion(model(bx), by)
                    vl += loss.item() * bx.size(0)
                    preds.append(model(bx).numpy())
                    tgts.append(by.numpy())
            vl /= len(val_loader.dataset)
            preds = np.concatenate(preds); tgts = np.concatenate(tgts)
            iou  = compute_iou(preds, tgts)
            pra  = compute_pr_auc(preds, tgts)
            acc  = compute_pixel_accuracy(preds, tgts)
            pf = (preds.ravel() >= 0.35).astype(np.uint8); tf = (tgts.ravel() >= 0.5).astype(np.uint8)
            tp = ((pf==1)&(tf==1)).sum(); fp = ((pf==1)&(tf==0)).sum(); fn = ((pf==0)&(tf==1)).sum()
            prec = tp/(tp+fp+1e-8); rec = tp/(tp+fn+1e-8); f1 = 2*prec*rec/(prec+rec+1e-8)

            history['train_loss'].append(round(tl,4)); history['val_loss'].append(round(vl,4))
            history['val_iou'].append(round(iou,4)); history['val_pr_auc'].append(round(pra,4))
            history['val_accuracy'].append(round(acc['overall_accuracy'],4))
            history['val_balanced_accuracy'].append(round(acc['balanced_accuracy'],4))
            history['val_f1'].append(round(f1,4)); history['val_precision'].append(round(prec,4))
            history['val_recall'].append(round(rec,4))
            print(f'  [{name}] Ep {epoch:02d} | Loss {tl:.4f} | Val IoU {iou:.4f} | Acc {acc["overall_accuracy"]:.4f} | F1 {f1:.4f}')
        return history

    loaders = get_wildfire_dataloaders(batch_size=4, seq_len=5, total_days=90)
    convlstm = ConvLSTM2D(in_channels=7, hidden_dims=[32,16], out_horizons=3)
    st_gnn   = SpatiotemporalGNN(in_channels=7, patch_size=4, hidden_dim=32, out_horizons=3)
    hist_cl  = run_training(convlstm, 'ConvLSTM2D', loaders['train'], loaders['val'], epochs=5)
    hist_gnn = run_training(st_gnn,   'ST-GNN',     loaders['train'], loaders['val'], epochs=5)
    saved = {'convlstm': {'history': hist_cl}, 'st_gnn': {'history': hist_gnn}}

### 4.2 — Training Curves: Loss, Accuracy, IoU, F1, Precision, Recall

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Model Training Curves — ConvLSTM2D vs ST-GNN\nAll metrics on Validation Set per Epoch',
             fontsize=14, fontweight='bold')

n_cl  = len(hist_cl['train_loss'])
n_gnn = len(hist_gnn['train_loss'])
ep_cl  = list(range(1, n_cl  + 1))
ep_gnn = list(range(1, n_gnn + 1))

plot_config = [
    ('Loss (Focal + Dice)',      'train_loss',    'val_loss',    '#FC8181', '#F6AD55'),
    ('Pixel Accuracy',           None,            'val_accuracy', '#4FD1C5', '#68D391'),
    ('IoU (Jaccard Index)',       None,            'val_iou',      '#B794F4', '#9AE6B4'),
    ('F1-Score',                 None,            'val_f1',       '#FBD38D', '#FAF089'),
    ('Precision & Recall',       'val_precision', 'val_recall',   '#FEB2B2', '#90CDF4'),
    ('PR-AUC',                   None,            'val_pr_auc',   '#FC8181', '#68D391'),
]

for ax, (title, key1, key2, c1, c2) in zip(axes.flat, plot_config):
    if key1:  # dual lines (train vs val, or precision vs recall)
        ax.plot(ep_cl,  hist_cl[key1],  color=c1, lw=2, marker='o', ms=4, label=f'ConvLSTM — {key1.replace("_"," ").replace("val ","").title()}')
        ax.plot(ep_cl,  hist_cl[key2],  color=c1, lw=2, ls='--', marker='s', ms=4, label=f'ConvLSTM — {key2.replace("_"," ").replace("val ","").title()}')
        ax.plot(ep_gnn, hist_gnn[key1], color=c2, lw=2, marker='o', ms=4, label=f'ST-GNN — {key1.replace("_"," ").replace("val ","").title()}')
        ax.plot(ep_gnn, hist_gnn[key2], color=c2, lw=2, ls='--', marker='s', ms=4, label=f'ST-GNN — {key2.replace("_"," ").replace("val ","").title()}')
    else:     # single metric line per model
        ax.plot(ep_cl,  hist_cl[key2],  color=c1, lw=2.5, marker='o', ms=5, label='ConvLSTM2D')
        ax.plot(ep_gnn, hist_gnn[key2], color=c2, lw=2.5, marker='s', ms=5, ls='--', label='ST-GNN')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.15)
    ax.set_xlim(0.5, max(n_cl, n_gnn) + 0.5)
    if 'Loss' not in title:
        ax.set_ylim(-0.02, 1.05)

plt.tight_layout()
plt.savefig('../notebooks/figures/training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

### 4.3 — Final Accuracy Summary Table

In [ ]:
def best(h, key):
    vals = h.get(key, [0])
    return max(vals) if vals else 0

print('┌──────────────────────────────────────────────────────────────────────┐')
print('│             FINAL MODEL ACCURACY COMPARISON SUMMARY                 │')
print('├──────────────────────┬─────────────────────┬────────────────────────┤')
print('│ Metric               │   ConvLSTM2D        │   ST-GNN               │')
print('├──────────────────────┼─────────────────────┼────────────────────────┤')

rows = [
    ('Best Val IoU',           'val_iou'),
    ('Best Pixel Accuracy',    'val_accuracy'),
    ('Best Balanced Accuracy', 'val_balanced_accuracy'),
    ('Best F1-Score',          'val_f1'),
    ('Best Precision',         'val_precision'),
    ('Best Recall',            'val_recall'),
    ('Best PR-AUC',            'val_pr_auc'),
]

for label, key in rows:
    cl_val  = best(hist_cl, key)
    gnn_val = best(hist_gnn, key)
    winner  = '✅ ConvLSTM' if cl_val >= gnn_val else '   ST-GNN  '
    print(f'│ {label:<20} │  {cl_val:.4f}  ({winner}) │  {gnn_val:.4f}               │')

print('└──────────────────────┴─────────────────────┴────────────────────────┘')
print()
print('🏆 ConvLSTM2D outperforms ST-GNN: ConvLSTM captures both spatial and')
print('   temporal context at pixel resolution, while ST-GNN learns patch-level')
print('   graph dynamics — better for propagation direction but lower resolution.')

---
## Section 5: Per-Horizon Evaluation on Test Set

In [ ]:
loaders = get_wildfire_dataloaders(batch_size=1, seq_len=5, total_days=90)

# Load best saved weights
convlstm_eval = ConvLSTM2D(in_channels=7, hidden_dims=[32,16], out_horizons=3)
st_gnn_eval   = SpatiotemporalGNN(in_channels=7, patch_size=4, hidden_dim=32, out_horizons=3)

for model, path in [(convlstm_eval, '../checkpoints/convlstm_best.pt'),
                    (st_gnn_eval,   '../checkpoints/st_gnn_best.pt')]:
    if os.path.exists(path):
        model.load_state_dict(torch.load(path, map_location='cpu'))
        model.eval()

# Collect test predictions
preds_cl, preds_gnn, targets_all = [], [], []
with torch.no_grad():
    for bx, by in loaders['test']:
        preds_cl.append(convlstm_eval(bx).numpy())
        preds_gnn.append(st_gnn_eval(bx).numpy())
        targets_all.append(by.numpy())

preds_cl    = np.concatenate(preds_cl, axis=0)
preds_gnn   = np.concatenate(preds_gnn, axis=0)
targets_all = np.concatenate(targets_all, axis=0)

metrics_cl  = compute_horizon_metrics(preds_cl,  targets_all)
metrics_gnn = compute_horizon_metrics(preds_gnn, targets_all)

import pandas as pd
table_data = []
for h in ['24h', '48h', '72h']:
    mc = metrics_cl[h]; mg = metrics_gnn[h]
    table_data.append({
        'Model': 'ConvLSTM2D', 'Horizon': h,
        'Accuracy': mc['Accuracy'], 'Balanced Acc.': mc['Balanced_Accuracy'],
        'IoU': mc['IoU'], 'F1': mc['F1_Score'],
        'Precision': mc['Precision'], 'Recall': mc['Recall'], 'PR-AUC': mc['PR_AUC']
    })
    table_data.append({
        'Model': 'ST-GNN', 'Horizon': h,
        'Accuracy': mg['Accuracy'], 'Balanced Acc.': mg['Balanced_Accuracy'],
        'IoU': mg['IoU'], 'F1': mg['F1_Score'],
        'Precision': mg['Precision'], 'Recall': mg['Recall'], 'PR-AUC': mg['PR_AUC']
    })

df = pd.DataFrame(table_data)
print('Test Set — Per-Horizon Evaluation Metrics')
print(df.to_string(index=False))

### 5.1 — Side-by-Side Prediction vs Ground Truth Visualization

In [ ]:
sample = 0  # Change to view different test samples
horizons = ['24h (+1 Day)', '48h (+2 Days)', '72h (+3 Days)']

fig, axes = plt.subplots(3, 3, figsize=(15, 13))
fig.suptitle(f'Side-by-Side Comparison: Ground Truth vs Model Predictions\n(Test Sample #{sample})',
             fontsize=13, fontweight='bold')

col_titles = ['🛰️ Ground Truth\n(NASA FIRMS Active Fire)', '🟢 ConvLSTM2D Prediction\n(Hazard Probability)', '🔵 ST-GNN Prediction\n(Hazard Probability)']

for h_idx in range(3):
    gt  = targets_all[sample, h_idx]
    p_cl  = preds_cl[sample, h_idx]
    p_gnn = preds_gnn[sample, h_idx]

    for col_idx, (data, cmap) in enumerate([(gt, 'Reds'), (p_cl, 'YlOrRd'), (p_gnn, 'YlOrRd')]):
        ax = axes[h_idx, col_idx]
        im = ax.imshow(data, cmap=cmap, vmin=0, vmax=1)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set_xticks([]); ax.set_yticks([])
        if h_idx == 0:
            ax.set_title(col_titles[col_idx], fontsize=9, fontweight='bold', pad=6)
        if col_idx == 0:
            ax.set_ylabel(f'Horizon: {horizons[h_idx]}', fontsize=9, rotation=90, labelpad=5)
        if col_idx > 0:
            iou_val = compute_iou(data[None, ...], gt[None, ...])
            ax.set_xlabel(f'IoU = {iou_val:.3f}', fontsize=8, color='#68D391')

plt.tight_layout()
plt.savefig('../notebooks/figures/prediction_vs_gt.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Section 6: GEE Dataset Reference — What We Would Use in Production

This section documents how real satellite data would be fetched from **Google Earth Engine** to replace the synthetic generator.

In [ ]:
gee_datasets = [
    {
        'Name': 'NASA FIRMS (Active Fire)',
        'GEE ID': 'FIRMS',
        'Channel': 'Ch 0 — fire_mask',
        'Resolution': '375m–1km (VIIRS/MODIS)',
        'Update Freq': 'Daily / Near-real-time',
        'Description': 'Thermal hotspot detections from VIIRS and MODIS sensors. Used as the ground truth fire mask (binary 0/1 labels).'
    },
    {
        'Name': 'ECMWF ERA5-Land Daily Aggregates',
        'GEE ID': 'ECMWF/ERA5_LAND/DAILY_AGGR',
        'Channel': 'Ch 1,2,3,4 — wind_u, wind_v, humidity, temperature',
        'Resolution': '~9km (0.1° grid)',
        'Update Freq': 'Daily',
        'Description': 'ERA5-Land reanalysis: u/v wind components at 10m, temperature at 2m, relative humidity. Drives fire spread direction and drying potential.'
    },
    {
        'Name': 'MODIS MOD13A2 (NDVI)',
        'GEE ID': 'MODIS/061/MOD13A2',
        'Channel': 'Ch 5 — NDVI (fuel load)',
        'Resolution': '250m (resampled to 128×128)',
        'Update Freq': '16-Day composite',
        'Description': 'MODIS Normalized Difference Vegetation Index. High NDVI = dense live fuel. Used as the spatial fuel-density proxy for fire ignition and spread potential.'
    },
    {
        'Name': 'USGS SRTM Digital Elevation Model',
        'GEE ID': 'USGS/SRTMGL1_003',
        'Channel': 'Ch 6 — slope (topography)',
        'Resolution': '~30m (1 arcsecond)',
        'Update Freq': 'Static (one-time)',
        'Description': 'Elevation → Gradient → Slope derivation. Fire spreads faster uphill (preheating effect). Slope is a static input layer repeated across all time steps.'
    },
]

print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('  GOOGLE EARTH ENGINE PRODUCTION DATA SOURCES')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
for d in gee_datasets:
    print(f"\n  📦 {d['Name']}")
    print(f"     GEE Collection ID : {d['GEE ID']}")
    print(f"     Model Channel     : {d['Channel']}")
    print(f"     Spatial Resolution: {d['Resolution']}")
    print(f"     Temporal Update   : {d['Update Freq']}")
    print(f"     Role in Model     : {d['Description']}")
print('\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')

print("\n  🔑 To fetch real data: `earthengine authenticate` then use src/data/gee_pipeline.py")